In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Paramètres
Te = 1.0
T = 50
sigma_theta = 0.05
sigma_r = 5
sigma_Q = 1.0

# Modèle de mouvement
F = np.array([[1, Te, 0, 0], [0, 1, 0, 0],
              [0, 0, 1, Te], [0, 0, 0, 1]])
Q = sigma_Q**2 * np.array([[Te**3/3, Te**2/2, 0, 0],
                           [Te**2/2, Te, 0, 0],
                           [0, 0, Te**3/3, Te**2/2],
                           [0, 0, Te**2/2, Te]])
# Fonction d'observation polaire
def h(x):
    px, _, py, _ = x
    return np.array([np.arctan2(py, px), np.sqrt(px**2 + py**2)])

# Jacobien
def jacobian_h(x):
    px, _, py, _ = x
    rho2 = max(px**2 + py**2, 1e-6)
    rho = np.sqrt(rho2)
    return np.array([[-py / rho2, 0, px / rho2, 0],
                     [px / rho, 0, py / rho, 0]])

# Initialisation
x_init = np.array([3, 40, -4, 20])
x_true = np.zeros((4, T))
x_est = np.zeros((4, T))
x_true[:, 0] = x_init
x_est[:, 0] = x_init
P = np.eye(4)
sensor_choices = []



# Simulation de la trajectoire
for k in range(1, T):
    x_true[:, k] = F @ x_true[:, k-1] + np.random.multivariate_normal(np.zeros(4), Q)

# Génération des observations de chaque capteur
y_all = np.zeros((3, 2, T))
for i in range(3):
    for k in range(T):
        y_all[i, :, k] = h(x_true[:, k]) + np.random.multivariate_normal(np.zeros(2), R_list[i])

# EKF avec sélection du capteur
for k in range(1, T):
    x_pred = F @ x_est[:, k-1]
    P_pred = F @ P @ F.T + Q
    Hk = jacobian_h(x_pred)

    best_info = -np.inf
    best_i = 0
    for i in range(3):
        R_i = R_list[i]
        S = Hk @ P_pred @ Hk.T + R_i
        S += 1e-6 * np.eye(2)
        info = 0.5 * np.log(np.linalg.det(S) / np.linalg.det(R_i))
        if info > best_info:
            best_info = info
            best_i = i

    sensor_choices.append(best_i)
    R_sel = R_list[best_i]
    y_obs = y_all[best_i, :, k]
    S = Hk @ P_pred @ Hk.T + R_sel
    S += 1e-6 * np.eye(2)
    K = P_pred @ Hk.T @ np.linalg.inv(S)
    x_est[:, k] = x_pred + K @ (y_obs - h(x_pred))
    P = (np.eye(4) - K @ Hk) @ P_pred

# Tableau du capteur choisi
sensor_table = pd.DataFrame({
    "Temps": np.arange(1, T),
    "Capteur choisi": [i + 1 for i in sensor_choices]
})
print(sensor_table)

# Tracé de la trajectoire
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_true[0], y=x_true[2], mode='lines', name='Trajectoire réelle'))
fig.add_trace(go.Scatter(x=x_est[0], y=x_est[2], mode='lines', name='Estimation EKF'))

for i in range(3):
    obs_px = y_all[i, 1] * np.cos(y_all[i, 0])
    obs_py = y_all[i, 1] * np.sin(y_all[i, 0])
    fig.add_trace(go.Scatter(x=obs_px, y=obs_py, mode='markers', name=f'Capteur {i+1}'))

fig.update_layout(title="Suivi EKF avec sélection adaptative du capteur",
                  xaxis_title="x", yaxis_title="y", width=900, height=500)
fig.show()


    Temps  Capteur choisi
0       1               2
1       2               2
2       3               2
3       4               2
4       5               2
5       6               2
6       7               2
7       8               2
8       9               2
9      10               2
10     11               2
11     12               2
12     13               2
13     14               2
14     15               2
15     16               2
16     17               2
17     18               2
18     19               2
19     20               2
20     21               2
21     22               2
22     23               2
23     24               2
24     25               2
25     26               2
26     27               2
27     28               2
28     29               2
29     30               2
30     31               2
31     32               2
32     33               2
33     34               2
34     35               2
35     36               2
36     37               2
37     38   

In [27]:
import plotly.graph_objects as go

# Angles réels, estimés, observés
theta_true = np.array([h(x_true[:, k])[0] for k in range(T)])
theta_est = np.array([h(x_est[:, k])[0] for k in range(T)])
theta_obs_c1 = y_all[0, 0, :]
theta_obs_c2 = y_all[1, 0, :]
theta_obs_c3 = y_all[2, 0, :]
theta_obs_sel = np.array([y_all[sensor_choices[k], 0, k] for k in range(T - 1)])
theta_obs_sel = np.insert(theta_obs_sel, 0, theta_obs_sel[0])  # aligner avec le temps

# Tracé
fig = go.Figure()
fig.add_trace(go.Scatter(y=theta_true, mode='lines', name='θ réel', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=theta_obs_sel, mode='lines', name='θ observé (capteur choisi)', line=dict(color='orange')))
fig.add_trace(go.Scatter(y=theta_est, mode='lines', name='θ estimé', line=dict(color='green')))
fig.add_trace(go.Scatter(y=theta_obs_c1, mode='lines', name='θ capteur 1', line=dict(dash='dot')))
fig.add_trace(go.Scatter(y=theta_obs_c2, mode='lines', name='θ capteur 2', line=dict(dash='dot')))
fig.add_trace(go.Scatter(y=theta_obs_c3, mode='lines', name='θ capteur 3', line=dict(dash='dot')))

fig.update_layout(title="Évolution de l'angle θ (bearing) pour chaque capteur",
                  xaxis_title="Temps (k)",
                  yaxis_title="θ (radians)",
                  
                  width=1000,
                  height=500)
fig.show()


In [23]:
import plotly.graph_objects as go

# S'assurer que les matrices R_list ont été définies
# (si tu les as déjà, ignore cette ligne)
R_list = [
    np.diag([sigma_theta**2 + np.random.uniform(0, 0.002),
             sigma_r**2 + np.random.uniform(0, 4)])
    for _ in range(3)
]

# Initialisation pour recalcul de P
P_mutual = np.eye(4)

# Stockage de l'information mutuelle
info_values = np.zeros((3, T))

for k in range(T):
    x_pred = F @ x_est[:, k - 1] if k > 0 else x_est[:, 0]
    P_pred = F @ P_mutual @ F.T + Q
    Hk = jacobian_h(x_pred)

    for i in range(3):
        R_i = R_list[i]
        S = Hk @ P_pred @ Hk.T + R_i
        S += 1e-6 * np.eye(2)
        info = 0.5 * np.log(np.linalg.det(S) / np.linalg.det(R_i))
        info_values[i, k] = info

    P_mutual = P_pred  # mise à jour locale

# Tracé avec Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(y=info_values[0], mode='lines', name='Capteur 1', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=info_values[1], mode='lines', name='Capteur 2', line=dict(color='orange')))
fig.add_trace(go.Scatter(y=info_values[2], mode='lines', name='Capteur 3', line=dict(color='green')))

fig.update_layout(title="Information mutuelle par capteur au cours du temps",
                  xaxis_title="Temps (k)",
                  yaxis_title="Information mutuelle (nats)",
                  
                  width=900,
                  height=400)
fig.show()
